### Import Dependencies

In [2]:
import os
import openai
from qdrant_client import QdrantClient
from langsmith import Client

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

### Download and example reference data point from LangSmith

In [3]:
ls_client = Client(api_key=os.getenv("LANGSMITH_API_KEY"))

In [4]:
dataset = ls_client.read_dataset(
    dataset_name="rag-evaluation-dataset"
)

In [5]:
dataset

Dataset(name='rag-evaluation-dataset', description='RAG evaluation dataset', data_type=<DataType.kv: 'kv'>, id=UUID('12ce3aef-5f0d-42b9-b799-93c845c47482'), created_at=datetime.datetime(2026, 10, 8, 14, 23, 26, 394446, tzinfo=TzInfo(0)), modified_at=datetime.datetime(2026, 10, 8, 14, 23, 26, 394446, tzinfo=TzInfo(0)), example_count=34, session_count=0, last_session_start_time=None, inputs_schema=None, outputs_schema=None, transformations=None, metadata={'runtime': {'sdk': 'langsmith-py', 'library': 'langsmith', 'runtime': 'python', 'platform': 'macOS-27.0.1-arm64-arm-64bit-Mach-O', 'sdk_version': '0.14.4', 'runtime_version': '3.14.7', 'langchain_version': None, 'py_implementation': 'CPython', 'langchain_core_version': None}})

In [6]:
list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[19].outputs

{'groumd_truth': 'To upgrade your gaming setup, we recommend the KOORUI 24" Curved Full HD Monitor for an immersive display, paired with the Skytech Shadow Gaming PC featuring an Intel Core i5 and RTX 3050 graphics card.',
 'reference_context_ids': ['B0BZ944K9M', 'B0BZJP91PF'],
 'reference_descriptions': ['KOORUI 24" Curved 60Hz Computer Monitor LED Monitor Full HD 1080P HDMI VGA, 1800R, Tilt Adjustment, Eye Care, Black 24N5C 【 FHD 1800R Curved Monitor 】- KOORUI using a 23.6\'\' VA screen with 178°wide viewing angle, delivers 1920 x 1080 at 60Hz breathtaking viewing quality. The 1800R degree curved design immerses you completely into the working landscape, screen curvature as the images appear to wrap around you for an in depth, immersive experience. 【 Wide Compatibility 】- KOORUI 24 inch monitor equipped with 1 HDMI ports and VGA port, easy access to your favorite devices with 1 cable solution, compatible with most laptop and PC. 【 Panoramic Viewing 】- With an VA panel, 3-sided micro-

In [7]:
list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[19].inputs

{'question': 'What products should I buy if I want to upgrade my gaming desk with a curved monitor and a matching mechanical-style gaming PC?'}

In [8]:
reference_input = list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[19].inputs
reference_output = list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[19].outputs

### RAG Pipeline

In [ ]:

def get_embedding(text, model="text-embedding-3-small"):    
    response = openai.embeddings.create(
        input=text,
        model=model,
    )

    return response.data[0].embedding

def retrieve_data(query, qdrant_client, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retieved_context.append(result.payload["description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context

def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}
"""

    return prompt

def generate_answer(prompt):

    response = openai.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[{"role": "system", "content": prompt}],
        reasoning_effort="none"
    )

    return response.choices[0].message.content

def rag_pipeline(question, top_k=5):

    qdrant_client = QdrantClient(url="http://localhost:6333")

    retrieved_context = retrieve_data(question, qdrant_client, top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    final_result = {
        "answer": answer,
        "question": question,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"],
        "similarity_scores": retrieved_context["similarity_scores"]
    }

    return final_result

In [ ]:
rag_pipeline("Can I get some charger?")

### RAGAS Metrics

In [ ]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy

In [ ]:
ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-5.4-mini"))
ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

In [ ]:
reference_input

In [ ]:
reference_output

In [ ]:
result = rag_pipeline(reference_input["question"])

In [ ]:
result

In [ ]:
async def ragas_context_precision_id_based(run, example):
    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"]
    )

    scorer = IDBasedContextPrecision()

    return await scorer.single_turn_ascore(sample)

In [ ]:
await ragas_context_precision_id_based(result, reference_output)

In [ ]:
async def ragas_context_recall_id_based(run, example):
    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"]
    )

    scorer = IDBasedContextRecall()

    return await scorer.single_turn_ascore(sample)

In [ ]:
await ragas_context_recall_id_based(result, reference_output)

In [ ]:
async def ragas_faithfulness(run, example):

    sample = SingleTurnSample(
            user_input=run["question"],
            response=run["answer"],
            retrieved_contexts=run["retrieved_context"]
        )

    scorer = Faithfulness(llm=ragas_llm)
    
    return await scorer.single_turn_ascore(sample)

In [ ]:
await ragas_faithfulness(result, "")

In [ ]:
async def ragas_relevancy(run, example):

    sample = SingleTurnSample(
            user_input=run["question"],
            response=run["answer"],
            retrieved_contexts=run["retrieved_context"]
        )

    scorer = ResponseRelevancy(llm=ragas_llm, embeddings=ragas_embeddings)
    
    return await scorer.single_turn_ascore(sample)

In [ ]:
await ragas_relevancy(result, "")